## `POPSRegressionEllipse`: ellipsoid posteriors by direct optimization

Comparing `BayesianRidge` (epistemic only), `POPSRegression`
(sampling-based POPS hypercube), `POPSRegressionEllipse` (uniform
ellipsoid posterior fit by direct minimization of the
generalization-error objective, mean frozen at the POPS pre-fit) and
`POPSRegressionEllipse` with the closed-form PAC-Bayes layer, on a
misspecified polynomial surrogate fit to low-noise data.

The ellipsoid bounds track the POPS hypercube bounds, but are obtained
by an interior-point optimization of the exact projected-ball
pushforward likelihood — no posterior sampling is involved. With
`pac_bayes=True` the reported bounds are the max/min over the ensemble
of ellipses within the 2σ range of the analytic (Laplace)
hyperposterior: strictly broader than the bare ellipse, decaying onto
it at rate N.

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from popsregression import POPSRegression, POPSRegressionEllipse

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def target_function(x):
    return (x**3 + 0.01 * x**4) * 0.1 + np.sin(x) * x * 10.0


def generate_data(N):
    x_train = np.sort(
        np.append(np.random.uniform(-1, 1, N), np.linspace(-1, 1, 2)) * 10
    )
    x_dense = np.linspace(-1.1, 1.1, 51) * 10
    y_dense = target_function(x_dense)

    p = PolynomialFeatures(degree=4, include_bias=True)
    X_train = p.fit_transform(x_train.reshape(-1, 1))
    X_dense = p.fit_transform(x_dense.reshape(-1, 1))
    y_train = target_function(x_train)
    return X_train, x_train, y_train, X_dense, x_dense, y_dense


def plot_panel(ax, x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=None, y_min=None):
    if y_max is not None and y_min is not None:
        ax.fill_between(x_dense, y_min, y_max, alpha=0.2, facecolor="0.5",
                        label="max/min")
    else:
        ax.fill_between(x_dense, y_pred - 4 * y_std, y_pred + 4 * y_std,
                        alpha=0.2, facecolor="0.5", label=r"$\pm4\sigma$")
    ax.fill_between(x_dense, y_pred - 2 * y_std, y_pred + 2 * y_std,
                    alpha=0.5, facecolor="C1", label=r"$\pm2\sigma$")
    ax.plot(x_dense, y_pred, "C1-", lw=4)
    ax.plot(x_train, y_train, "b.", label="Train")
    ax.plot(x_dense, y_dense, "k-")

### BayesianRidge vs POPS Hypercube vs POPS Ellipse vs POPS Ellipse + PAC

Fitting a quartic polynomial (P=5) to a complex oscillatory function at
N = 10, 50, 500 training points. `BayesianRidge` epistemic uncertainty
vanishes with more data; the POPS variants maintain honest uncertainty
where the polynomial deviates from the truth. Both ellipse rows keep
the mean at the POPS pre-fit (`optimize_center=False`, the default) and
optimize only the widths; the PAC row widens the bounds to the 2σ
hyperposterior ensemble — most visibly in the scarce-data column,
where the ellipsoid itself is least determined.

In [ ]:
np.random.seed(42)

titles = ["Bayesian Ridge", "POPS Hypercube", "POPS Ellipse",
          "POPS Ellipse + PAC"]
fig, axs = plt.subplots(4, 3, figsize=(8, 9.5), sharex=True, sharey=True)
N_array = [10, 50, 500]

for i, N in enumerate(N_array):
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)

    bay = BayesianRidge(fit_intercept=False)
    hyc = POPSRegression(leverage_percentile=0.0, posterior="hypercube")
    ell = POPSRegressionEllipse(random_state=0)
    pac = POPSRegressionEllipse(random_state=0, pac_bayes=True)

    bay.fit(X_train, y_train)
    hyc.fit(X_train, y_train)
    ell.fit(X_train, y_train)
    pac.fit(X_train, y_train)

    # BayesianRidge - epistemic only (no aleatoric alpha_)
    b_pred = bay.predict(X_dense, return_std=False)
    b_std = np.sqrt(np.sum(np.dot(X_dense, bay.sigma_) * X_dense, axis=1))
    plot_panel(axs[0, i], x_dense, y_dense, x_train, y_train, b_pred, b_std)

    # POPS Hypercube (sampling-based)
    y_pred, y_std, y_max, y_min = hyc.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[1, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    # POPS Ellipse (direct optimization; bounds = ellipse max/min)
    y_pred, y_std, y_max, y_min = ell.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[2, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    # POPS Ellipse + PAC (bounds = 2-sigma hyperposterior ensemble)
    y_pred, y_std, y_max, y_min = pac.predict(
        X_dense, return_std=True, return_bounds=True
    )
    plot_panel(axs[3, i], x_dense, y_dense, x_train, y_train, y_pred, y_std,
               y_max=y_max, y_min=y_min)

    axs[0, i].set_title(f"N = {N}")

for j, title in enumerate(titles):
    axs[j, 0].set_ylim(-250, 250)
    axs[j, 0].set_xlim(-10, 10)
    axs[j, 1].legend(fontsize=9, loc="lower center")
    axs[j, 0].set_ylabel(title, fontsize=12)

plt.tight_layout()
plt.show()

### Observations

- **N/P = 2**: all methods show wide uncertainty. The bare ellipse row
  is deliberately tighter than the hypercube: it is the minimum-width
  covering support found by optimization, while the hypercube min/max
  is the support of an un-optimized box. The **PAC row restores
  conservatism** where it matters — its bounds are the max/min over
  the 2σ hyperposterior ensemble, and at N = 10 they cover the true
  function at roughly 70% of the hypercube's width.
- **N/P = 10 and 100**: BayesianRidge epistemic uncertainty collapses,
  while all POPS variants maintain honest uncertainty; the PAC bounds
  concentrate onto the bare ellipse bounds at rate N.
- Both ellipse rows keep `coef_` at the POPS/BayesianRidge pre-fit
  (`optimize_center=False` is the default). Joint optimization of the
  center (`optimize_center=True`) gives a tighter heteroscedastic-WLS
  fit — see the estimator docstring — at the cost of low-N
  conservatism.
- Every training point is covered by construction
  (`coverage_fraction_ == 1`).

### Closed-form PAC-Bayes layer

With `pac_bayes=True` the fit adds a diagonal Laplace hyperposterior,
giving closed-form KL and PAC-bound components — still without any
sampling.

With the default `hyperprior_center='phase1'` the hyperprior is centered
on the phase-1 optimum itself, so `pac_bayes=True` **never changes the
fitted ellipsoid** (`coef_`, `U_` are identical to the bare fit). The
hyperposterior — the epistemic uncertainty *about the ellipsoid* —
enters prediction analytically: `return_std` averages the pushforward
over it, and `return_bounds` returns the max/min over the **ensemble of
ellipses within the 2σ range of the hyperposterior**,
`mean ± (sqrt(v) + 2·y_bound_std)` — strictly broader than the bare
ellipse's support, which is recovered as `bounds ∓ 2·y_bound_std`. The
broadening is largest at N/P ~ 2 and decays at rate N as the
hyperposterior concentrates on the bare values.

`hyperprior_scale` is *relative*: the effective hyperprior variance is
`hyperprior_scale * ||psi_0||^2 / d`, so the default is independent of
the units of `y`. `hyperprior_center='warm_start'` instead centers the
prior on the POPS warm start (the fit is then ridge-shrunk toward the
baseline ellipsoid) and enables the Tipping/MacKay evidence update
`update_hyperprior=True`.

In [ ]:
for N in [10, 50, 500]:
    X_train, x_train, y_train, X_dense, x_dense, y_dense = generate_data(N)
    pac = POPSRegressionEllipse(random_state=0, pac_bayes=True)
    pac.fit(X_train, y_train)

    # PAC bounds (2-sigma hyperposterior ensemble) vs bare ellipse bounds
    _, p_max, p_min, p_bstd = pac.predict(
        X_dense, return_bounds=True, return_bound_std=True
    )
    bare_width = (p_max - p_min) - 4.0 * p_bstd
    broadening = np.mean((p_max - p_min) / bare_width - 1.0)

    d = pac.hyper_sigma_diag_.size
    print(
        f"N={N:<4d} coverage={pac.coverage_fraction_:.2f}  "
        f"G_hat={pac.objective_:6.3f}  kl/N={pac.kl_ / len(y_train):5.2f}  "
        f"bound={pac.bound_:6.3f}  gamma={pac.gamma_:5.1f} of {d}  "
        f"2sig broadening=+{100 * broadening:.1f}%"
    )

The bound tightens monotonically with N while every training point stays
covered, and the broadening over the bare bounds — strictly positive by
construction — decays as the hyperposterior concentrates on the bare
values at rate N: exactly the behaviour the hierarchical PAC-Bayes
construction prescribes.